# endgame-probe -- how does eat-rest-v1 die?

`tune_v1` found v1's numeric settings flat (80 variants, none better on fresh seeds), so the next gain has to come from a
behaviour change. This runs `external/candidates/endgame_probe.py`: the untouched baseline on fresh seeds (11000+), recording
a 50 s timeline of the world and the colony plus every death and birth, and prints what changes in the run-up to extinction.

16 games, roughly 5-8 minutes; runs in the foreground so the report lands in the cell. Resumable: rerunning skips finished
games, and raising `SEEDS` only plays the new ones. Mechanism study: nothing is written to `results/`.

**Cluster setup:** same as `parameter-tuning.ipynb` (`.env` with `GITHUB_TOKEN=<token>`).

In [ ]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}

In [ ]:
import glob
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

# Must run from survival-simulator/ so `src`, `agents`, `training` import.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])

print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

## Run the probe and print the report

In [ ]:
SEEDS = 16
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/endgame --seeds {SEEDS} 2>&1 | grep -v "pkg_resources\|pygame"

## Same seeds with a changed setting (optional)

Overrides go on top of the shipped config and the games land in their own folder, so the two reports can be compared seed by seed.

In [ ]:
OVERRIDES = '{"population": 4}'   # example only
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/endgame_alt --seeds {SEEDS} --set '{OVERRIDES}' 2>&1 | grep -v "pkg_resources\|pygame"

## Report only (no new games)

In [ ]:
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/endgame --seeds 0 2>&1 | grep -v "pkg_resources\|pygame"